<a href="https://colab.research.google.com/github/nerd2ninja/Colab-notebooks/blob/main/whisperx.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Core system deps
!apt-get update
!apt-get install -y ffmpeg

# PyTorch (CUDA 11.8 works well with T4)
!pip install -q torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118

# WhisperX
!pip install -q git+https://github.com/m-bain/whisperX.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of pyannote-core to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 10.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 11.1 MB/s eta 0:00:00
INFO: pip is still looking at multiple versions of pyannote-core to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 10.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 11.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2

In [1]:
from google.colab import files
uploaded = files.upload()

audio_file = list(uploaded.keys())[0]

Saving test.wav to test.wav


In [2]:
import whisperx
import torch
import json

device = "cuda"
compute_type = "float16"

# 🔥 High-quality model
model = whisperx.load_model("large-v2", device, compute_type=compute_type)

# Step 1: Transcribe
result = model.transcribe(audio_file)

# Step 2: Align (word-level precision)
model_a, metadata = whisperx.load_align_model(
    language_code=result["language"],
    device=device
)

result = whisperx.align(
    result["segments"],
    model_a,
    metadata,
    audio_file,
    device
)

# Step 3: Build clean sentence segments
sentence_segments = []

for seg in result["segments"]:
    words = seg.get("words", [])
    if not words:
        continue

    sentence_segments.append({
        "start": words[0]["start"],
        "end": words[-1]["end"],
        "text": seg["text"].strip()
    })

# Print preview
print(json.dumps(sentence_segments[:5], indent=2, ensure_ascii=False))

AttributeError: module 'torchaudio' has no attribute 'set_audio_backend'